# Visualize node id positions in 3D

Plot x,y,z positions of one or more lists of node ids from the `nbS1` circuit as an interactive 3D scatter. Each list becomes its own trace (colour + legend entry), so different neuron groups can be overlaid for comparison.

Requires `plotly` (`uv pip install plotly` or `pip install plotly`).

In [ ]:
import numpy as np
import plotly.graph_objects as go
from bluepysnap import Circuit
from pathlib import Path

In [ ]:
CIRCUIT_PATH = Path("/Users/james/Documents/obi/Data/Circuits/nbS1/circuit_config.json")
POPULATION = "S1nonbarrel_neurons"

circuit = Circuit(CIRCUIT_PATH)
nodes = circuit.nodes[POPULATION]
print(f"Population: {POPULATION} ({nodes.size} cells)")
print(f"Populations available: {circuit.nodes.population_names}")

## Define lists of node ids

Populate `node_id_lists` as `{label: ids}`. Each label becomes a legend entry in the 3D plot. Ids must belong to `POPULATION`.

In [ ]:
rng = np.random.default_rng(0)
all_ids = nodes.ids()

# Example: three random samples. Replace with your own lists.
node_id_lists = {
    "group_a": rng.choice(all_ids, size=500, replace=False),
    "group_b": rng.choice(all_ids, size=500, replace=False),
    "group_c": rng.choice(all_ids, size=500, replace=False),
}

# Alternative: pull ids from named node sets.
# node_id_lists = {
#     "L2_TPC:A": nodes.ids("L2_TPC:A"),
#     "L5_TPC:A": nodes.ids("L5_TPC:A"),
# }

In [ ]:
def get_positions(population, node_ids):
    """Return (N, 3) x,y,z positions for the given node ids, in input order."""
    ids = [int(i) for i in node_ids]
    df = population.get(ids, properties=["x", "y", "z"])
    return df.loc[ids].to_numpy(dtype=float)


positions = {label: get_positions(nodes, ids) for label, ids in node_id_lists.items()}
for label, xyz in positions.items():
    print(f"{label}: {xyz.shape[0]} points")

In [ ]:
def plot_node_positions_3d(
    positions_by_label, *, marker_size=3, title=None, background=None
):
    """Interactive 3D scatter. One trace per label; optional faint background cloud."""
    fig = go.Figure()
    if background is not None:
        fig.add_trace(
            go.Scatter3d(
                x=background[:, 0],
                y=background[:, 1],
                z=background[:, 2],
                mode="markers",
                marker={"size": 1, "color": "lightgrey", "opacity": 0.3},
                name=f"background (n={background.shape[0]})",
                hoverinfo="skip",
            )
        )
    for label, xyz in positions_by_label.items():
        fig.add_trace(
            go.Scatter3d(
                x=xyz[:, 0],
                y=xyz[:, 1],
                z=xyz[:, 2],
                mode="markers",
                marker={"size": marker_size},
                name=f"{label} (n={xyz.shape[0]})",
            )
        )
    fig.update_layout(
        title=title,
        scene={
            "xaxis_title": "x (µm)",
            "yaxis_title": "y (µm)",
            "zaxis_title": "z (µm)",
            "aspectmode": "data",
        },
        legend={"itemsizing": "constant"},
        margin={"l": 0, "r": 0, "t": 40, "b": 0},
    )
    return fig


fig = plot_node_positions_3d(positions, title=f"{POPULATION} node positions")
fig.show()

## Optional: with a sub-sampled background for spatial context

In [ ]:
bg_sample = rng.choice(all_ids, size=min(5000, nodes.size), replace=False)
background_xyz = get_positions(nodes, bg_sample)

fig = plot_node_positions_3d(
    positions,
    title=f"{POPULATION} node positions (with background)",
    background=background_xyz,
)
fig.show()